In [ ]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent 
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
duck = connect_to_postgres_via_duckdb()

In [ ]:
create_clean_account_name_macro(duck)

# create schema and dashboard table

In [ ]:
duck.sql(
    """
        select
            "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden", 'berlin' as city
    from read_csv('../merge_tables/data/berlin_active_contracts.csv', skip=4)
"""
)

In [ ]:
duck.sql(
    """
    select * exclude (column00, column01, column02), 'viersen' as city
    from read_csv('../merge_tables/data/viersen_active_contracts.csv', skip=4)

    """
)


In [ ]:
duck.sql(
    f"""
    create or replace table easybill_dashboard as 
    select 
        row_number() over () || 'berlin' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'berlin' as city
    from read_csv('../merge_tables/data/berlin_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select 
        row_number() over() || 'dusseldorf' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'dusseldorf' as city
    from read_csv('../merge_tables/data/dusseldorf_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select 
        row_number() over() || 'frankfurt' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'frankfurt' as city
    from read_csv('../merge_tables/data/frankfurt_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select 
        row_number() over() || 'hamburg' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'hamburg' as city
    from read_csv('../merge_tables/data/hamburg_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select 
        row_number() over() || 'kiel' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'kiel' as city
    from read_csv('../merge_tables/data/kiel_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select
        row_number() over() || 'koln' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'koln' as city
    from read_csv('../merge_tables/data/koln_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select
        row_number() over() || 'munchen' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'munchen' as city
    from read_csv('../merge_tables/data/munchen_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select
        row_number() over() || 'rostock' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'rostock' as city
    from read_csv('../merge_tables/data/rostock_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select
        row_number() over() || 'stuttgart' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, BG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'stuttgart' as city
    from read_csv('../merge_tables/data/stuttgart_active_contracts.csv', skip=4)
    where "Unternehmen" is not null

    union all
    select
        row_number() over() || 'viersen' as id,
        "#", "Unternehmen" , "Kd-Nr.", Arzt, FaSi, Betreuungsart, Mitarbeiter, null asBG, "Vertrags-\nbeginn", "Vertrags-\nstunden",
        'viersen' as city
    from read_csv('../merge_tables/data/viersen_active_contracts.csv', skip=4)
    where "Unternehmen" is not null
    
    
    """
)

In [ ]:
duck.execute(
    """

    begin;
    create schema if not exists pg.easybill;

    create or replace table pg.easybill.dashboard as select * from easybill_dashboard;
    
    commit;
    """
)

# create tables



# add Kontact and Posten tables


In [ ]:
duck.sql(
    """
    create table if not exists pg.easybill.contacts as
    select 
        distinct on ("Kontakt: Kontakt ID") 
        * like 'Kontakt: %'
    from read_csv('/Users/adrienblanquer/Downloads/Documents-Export-21_01_2026-14_38_44.csv', types={'Kontakt: Kundennummer': 'VARCHAR'})
    """
)

In [ ]:
duck.sql(
    """
    create or replace table pg.easybill.posten as
    select 
        --distinct on ("Kontakt: Kontakt ID") 
        "Kontakt: Kundennummer",
        * like 'Posten: %'
    from read_csv('/Users/adrienblanquer/Downloads/Documents-Export-21_01_2026-14_38_44.csv', types={'Kontakt: Kundennummer': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Posten: ID': 'VARCHAR'})

    """
)

# table de jointure medisoft x zoho x easybill

In [ ]:
duck.sql(
    f"""
    create or replace table deutshland_consolidated as (
        select *, 'berlin' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "berlin_firms_consolidated.csv"}')
      union all 
      select *, 'dusseldorf' as city
      from read_csv('../merge_tables/{OUTPUT_DIR / "dusseldorf_firms_consolidated.csv"}')
        union all 
        select *, 'frankfurt' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "frankfurt_firms_consolidated.csv"}')
        union all 
        select *, 'hamburg' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "hamburg_firms_consolidated.csv"}')
        union all 
        select *, 'kiel' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "kiel_firms_consolidated.csv"}')
        union all 
        select *, 'koln' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "koln_firms_consolidated.csv"}')
        union all 
        select *, 'munchen' as city 
        from read_csv('../merge_tables/{OUTPUT_DIR / "munchen_firms_consolidated.csv"}')
        union all 
        select *, 'rostock' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "rostock_firms_consolidated.csv"}')
        union all 
        select *, 'stuttgart' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "stuttgart_firms_consolidated.csv"}')
        union all
        select *, 'viersen' as city
        from read_csv('../merge_tables/{OUTPUT_DIR / "viersen_firms_consolidated.csv"}')       
    )
    """
    )

In [ ]:
duck.sql("from deutshland_consolidated")

In [ ]:
duck.execute(
    """
    create or replace table pg.easybill.easybill_x_medisoft as 
    from deutshland_consolidated
    """
)

In [ ]:
duck.sql("""select distinct left("Kd-Nr."::varchar, 9) from pg.easybill.dashboard """)

In [ ]:
duck.sql("select distinct id_easybill from pg.easybill.easybill_x_medisoft")

In [ ]:
duck.sql(
    """
    select
        *
    from pg.easybill.easybill_x_medisoft
    left join pg.medisoft.table_firms_zoho
    on pg.easybill.easybill_x_medisoft.id_medisoft = pg.medisoft.table_firms_zoho.rec_id
    """
)

# Match entre les trois bases (en partant du dashboard easybill) > Donc les firms actives

In [ ]:
duck.sql(
    """
    create or replace table pg.easybill.easybill_x_zoho_x_medisoft as 
    with easybill_data as (
        select 
            id as source_id, left("Kd-Nr."::varchar, 9) as id_easybill,
            clean_account_name(Unternehmen) as clean_name,
            Unternehmen as name,
        from pg.easybill.dashboard as active_accounts
    ), medisoft_data as (
        select 
            rec_id as source_id,
            rec_id as id_medisoft,
            clean_account_name(coalesce(split(pfad, '/')[2], split(pfad, '/')[1], name, kuerzel)) as medisoft_clean_name,
            coalesce(split(pfad, '/')[2], split(pfad, '/')[1], name, kuerzel) as medisoft_name,
        from pg.medisoft.table_firmenstruktur
    ), eb_x_medisoft_matched_firms as (
        select
            *,
            jaro_winkler_similarity(eb.clean_name, me.medisoft_clean_name) as sim
        from easybill_data eb
        left join medisoft_data me
            on eb.clean_name = me.medisoft_clean_name
            or jaro_winkler_similarity(eb.clean_name, me.medisoft_clean_name) > 0.95
        where sim > 0.6
        qualify row_number() over (partition by me.id_medisoft order by sim desc) = 1
        order by eb.id_easybill
    ), zoho_data as (
        select
            Id as source_id,
            Id as id_zoho,
            clean_account_name(Account_Name) as zoho_clean_name,
            Account_Name as zoho_name,
            Employees as zoho_employees,
        from pg.zoho.Accounts
    ), eb_x_zoho_matched_firms as (
        select
            *,
            jaro_winkler_similarity(eb.clean_name, zo.zoho_clean_name) as sim
        from easybill_data eb
        join zoho_data zo
            on eb.clean_name = zo.zoho_clean_name
            or jaro_winkler_similarity(eb.clean_name, zo.zoho_clean_name) > 0.95
        qualify row_number() over (partition by eb.id_easybill order by sim desc, zoho_employees desc) = 1
        order by eb.id_easybill
    )
    select 
        eb_x_zoho_matched_firms.id_easybill as id_easybill,
        id_zoho as id_zoho,
        id_medisoft as id_medisoft,
        eb_x_zoho_matched_firms.name as easybill_name,
        eb_x_zoho_matched_firms.clean_name as easybill_clean_name,
        zoho_name as zoho_name,
        zoho_clean_name as zoho_clean_name,
        medisoft_name as medisoft_name,
        medisoft_clean_name as medisoft_clean_name,
    from eb_x_zoho_matched_firms
    left join eb_x_medisoft_matched_firms
        on eb_x_zoho_matched_firms.id_easybill = eb_x_medisoft_matched_firms.id_easybill
    --where 'kline' in eb_x_zoho_matched_firms.clean_name 
    """
)

# Match entre les trois bases, en partant cette fois de la liste de tous les clients easybill

In [ ]:
from merge_tables.db.tables import create_clean_account_name_macro
create_clean_account_name_macro(duck)

In [ ]:
duck.sql(
    """
    create or replace table pg.easybill.all_easybill_x_zoho_x_medisoft as 
    with easybill_data as (
        select 
            "Kontakt: Kundennummer" as id_easybill,
            "Kontakt: Firma" as easybill_name,
            clean_account_name("Kontakt: Firma") as easybill_clean_name
        from pg.easybill.contacts
        where "Kontakt: Firma" is not null -- on prend que les firs qui ont un nom
    ), medisoft_data as (
        select 
            rec_id as source_id,
            rec_id as id_medisoft,
            clean_account_name(
                case 
                    when array_length(split(pfad, '/')) >= 3 and trim(split(pfad, '/')[2]) = 'Nicht Kunden' 
                        then coalesce(
                            nullif(split(pfad, '/')[3], ''), 
                            nullif(name, ''), 
                            kuerzel
                        )
                    else coalesce(
                            nullif(split(pfad, '/')[2], ''), 
                            nullif(split(pfad, '/')[1], ''), 
                            nullif(name, ''), 
                            kuerzel
                        )
                end
            ) as medisoft_clean_name,
            trim(case 
                when array_length(split(pfad, '/')) >= 3 and trim(split(pfad, '/')[2]) = 'Nicht Kunden' 
                    then coalesce(
                        nullif(split(pfad, '/')[3], ''), 
                        nullif(name, ''), 
                        kuerzel
                    )
                else coalesce(
                        nullif(split(pfad, '/')[2], ''), 
                        nullif(split(pfad, '/')[1], ''), 
                        nullif(name, ''), 
                        kuerzel
                    )
            end) as medisoft_name,
        from pg.medisoft.table_firmenstruktur
    ), eb_x_medisoft_matched_firms as (
        select
            *,
            jaro_winkler_similarity(eb.easybill_clean_name, me.medisoft_clean_name) as sim
        from easybill_data eb
        left join medisoft_data me
            on eb.easybill_clean_name = me.medisoft_clean_name
            or jaro_winkler_similarity(eb.easybill_clean_name, me.medisoft_clean_name) > 0.95
        where sim > 0.6
        qualify row_number() over (partition by me.id_medisoft order by sim desc) = 1
        order by eb.id_easybill
    ), zoho_data as (
        select
            Id as source_id,
            Id as id_zoho,
            clean_account_name(Account_Name) as zoho_clean_name,
            Account_Name as zoho_name,
            Employees as zoho_employees,
        from pg.zoho.Accounts
    ), eb_x_zoho_matched_firms as (
        select
            *,
            jaro_winkler_similarity(eb.easybill_clean_name, zo.zoho_clean_name) as sim
        from easybill_data eb
        left join zoho_data zo
            on eb.easybill_clean_name = zo.zoho_clean_name
            or jaro_winkler_similarity(eb.easybill_clean_name, zo.zoho_clean_name) > 0.95
        qualify row_number() over (partition by eb.id_easybill order by sim desc, zoho_employees desc) = 1
        order by eb.id_easybill
    ), first_pass as (
        select
            ez.id_easybill,
            ez.id_zoho,
            em.id_medisoft,
            ez.easybill_name,
            ez.easybill_clean_name,
            ez.zoho_name,
            ez.zoho_clean_name,
            em.medisoft_name,
            em.medisoft_clean_name
        from eb_x_zoho_matched_firms ez
        left join eb_x_medisoft_matched_firms em on ez.id_easybill = em.id_easybill
    ), medisoft_data_2 as (
        select
            rec_id as source_id,
            rec_id as id_medisoft,
            clean_account_name(coalesce(name, kuerzel)) as medisoft_clean_name,
            coalesce(name, kuerzel) as medisoft_name
        from pg.medisoft.table_firmenstruktur
        where rec_id not in (select distinct id_medisoft from first_pass where id_medisoft is not null)
    ), eb_x_medisoft_matched_firms_2 as (
        select
            fp.id_easybill,
            me.id_medisoft,
            me.medisoft_name,
            me.medisoft_clean_name,
            jaro_winkler_similarity(fp.easybill_clean_name, me.medisoft_clean_name) as sim
        from first_pass fp
        inner join medisoft_data_2 me
            on fp.easybill_clean_name = me.medisoft_clean_name
            or jaro_winkler_similarity(fp.easybill_clean_name, me.medisoft_clean_name) > 0.95
        where fp.id_medisoft is null
        qualify row_number() over (partition by fp.id_easybill order by sim desc) = 1
    )
    select
        fp.id_easybill,
        fp.id_zoho,
        coalesce(fp.id_medisoft, m2.id_medisoft) as id_medisoft,
        fp.easybill_name,
        fp.easybill_clean_name,
        fp.zoho_name,
        fp.zoho_clean_name,
        coalesce(fp.medisoft_name, m2.medisoft_name) as medisoft_name,
        coalesce(fp.medisoft_clean_name, m2.medisoft_clean_name) as medisoft_clean_name
    from first_pass fp
    left join eb_x_medisoft_matched_firms_2 m2 on fp.id_easybill = m2.id_easybill
    """
)

In [ ]:
duck.sql(
    """
    --create or replace table pg.easybill.all_easybill_x_zoho_x_medisoft as 
    with easybill_data as (
        select 
            "Kontakt: Kundennummer" as id_easybill,
            "Kontakt: Firma" as easybill_name,
            clean_account_name("Kontakt: Firma") as easybill_clean_name
        from pg.easybill.contacts
        where "Kontakt: Firma" is not null -- on prend que les firs qui ont un nom
    ), medisoft_data as (
        select 
            rec_id as source_id,
            rec_id as id_medisoft,
            clean_account_name(
                case 
                    when array_length(split(pfad, '/')) >= 3 and trim(split(pfad, '/')[2]) = 'Nicht Kunden' 
                        then coalesce(
                            nullif(split(pfad, '/')[3], ''), 
                            nullif(name, ''), 
                            kuerzel
                        )
                    else coalesce(
                            nullif(split(pfad, '/')[2], ''), 
                            nullif(split(pfad, '/')[1], ''), 
                            nullif(name, ''), 
                            kuerzel
                        )
                end
            ) as medisoft_clean_name,
            trim(case 
                when array_length(split(pfad, '/')) >= 3 and trim(split(pfad, '/')[2]) = 'Nicht Kunden' 
                    then coalesce(
                        nullif(split(pfad, '/')[3], ''), 
                        nullif(name, ''), 
                        kuerzel
                    )
                else coalesce(
                        nullif(split(pfad, '/')[2], ''), 
                        nullif(split(pfad, '/')[1], ''), 
                        nullif(name, ''), 
                        kuerzel
                    )
            end) as medisoft_name,
        from pg.medisoft.table_firmenstruktur
    ), eb_x_medisoft_matched_firms as (
        select
            *,
            jaro_winkler_similarity(eb.easybill_clean_name, me.medisoft_clean_name) as sim
        from easybill_data eb
        left join medisoft_data me
            on eb.easybill_clean_name = me.medisoft_clean_name
            or jaro_winkler_similarity(eb.easybill_clean_name, me.medisoft_clean_name) > 0.95
        where sim > 0.6
        qualify row_number() over (partition by eb.id_medisoft order by sim desc) = 1
        order by eb.id_easybill
    ), zoho_data as (
        select
            Id as source_id,
            Id as id_zoho,
            clean_account_name(Account_Name) as zoho_clean_name,
            Account_Name as zoho_name,
            Employees as zoho_employees,
        from pg.zoho.Accounts
    ), eb_x_zoho_matched_firms as (
        select
            *,
            jaro_winkler_similarity(eb.easybill_clean_name, zo.zoho_clean_name) as sim
        from easybill_data eb
        left join zoho_data zo
            on eb.easybill_clean_name = zo.zoho_clean_name
            or jaro_winkler_similarity(eb.easybill_clean_name, zo.zoho_clean_name) > 0.95
        qualify row_number() over (partition by eb.id_easybill order by sim desc, zoho_employees desc) = 1
        order by eb.id_easybill
    )
    --select count(*) from (
        select 
            eb_x_zoho_matched_firms.id_easybill as id_easybill,
            id_zoho as id_zoho,
            id_medisoft as id_medisoft,
            eb_x_zoho_matched_firms.easybill_name as easybill_name,
            eb_x_zoho_matched_firms.easybill_clean_name as easybill_clean_name,
            zoho_name as zoho_name,
            zoho_clean_name as zoho_clean_name,
            medisoft_name as medisoft_name,
            medisoft_clean_name as medisoft_clean_name,
        from eb_x_zoho_matched_firms
        left join eb_x_medisoft_matched_firms
            on eb_x_zoho_matched_firms.id_easybill = eb_x_medisoft_matched_firms.id_easybill
        where 'ejf' in eb_x_zoho_matched_firms.easybill_clean_name
       -- ) where id_zoho is not null
    """
)

In [ ]:
duck.sql("""from pg.easybill.contacts where "Kontakt: Firma" is not null""")

In [ ]:
duck.execute("""
CREATE OR REPLACE MACRO clean_account_name(str) AS (
    SELECT 
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    lower(
                        strip_accents(
                            replace(str, '&#38;', '')
                        )
                    ), 
                    -- 1. Suffixes juridiques et tout ce qui suit
                    '(\\b(\\s|\\))+(gmbh|mbh|g?ag|ev|kg|kgaa|se|llp|ek|ohg|ug|inc|ltd|corp|plc|gemeinnutzige)\\b).*$',
                    ''
                ),
                -- 2. Villes allemandes et variantes (nettoyées par strip_accents et lower)
                -- L'ordre privilégie les noms composés (ex: frankfurt am main) avant les noms simples
                '\\b(' ||
                -- Autour de Berlin
                'berlin|potsdam|cottbus|' ||
                -- Zone Rhénanie / Ruhr (Düsseldorf, Köln, Viersen...)
                'dusseldorf|koln|cologne|bonn|dortmund|essen|duisburg|bochum|wuppertal|monchengladbach|gelsenkirchen|aachen|krefeld|oberhausen|leverkusen|neuss|viersen|' ||
                -- Sud (Stuttgart, München...)
                'stuttgart|karlsruhe|mannheim|freiburg|heidelberg|heilbronn|muenchen|munchen|munich|augsburg|nurnberg|nuremberg|regensburg|' ||
                -- Nord (Hamburg, Kiel, Rostock...)
                'hamburg 1|hamburg|bremen|kiel|lubeck|flensburg|rostock|schwerin|' ||
                -- Centre (Frankfurt, Hannover...)
                'frankfurt am main|frankfurt|wiesbaden|mainz|darmstadt|offenbach|hannover|braunschweig|wolfsburg|kassel|' ||
                -- Divers
                'ueberregional|leipzig|dresden|chemnitz|magdeburg|halle|erfurt|' ||
                'germany' ||
                ')\\b',
                '',
                'g'
            ),
            -- 3. Nettoyage final (caractères spéciaux et espaces doubles)
            '[^a-z0-9]',
            '',
            'g'
        )
);
""")


In [ ]:
duck.sql(""" 
select clean_account_name('Avanti GmbH')
""")

# Stats before

In [ ]:
duck.sql(
    """
    select 
        count(*) filter(where id_medisoft is not null and id_zoho is not null) as both_matched,
        count(*) filter(where id_medisoft is not null) as medisoft_matched,
        count(*) filter(id_zoho is not null) as zoho_matched,
    from pg.easybill.all_easybill_x_zoho_x_medisoft
    """)

# Stats after

In [ ]:
duck.sql(
    """
    select 
        count(*) filter(where id_medisoft is not null and id_zoho is not null) as both_matched,
        count(*) filter(where id_medisoft is not null) as medisoft_matched,
        count(*) filter(id_zoho is not null) as zoho_matched,
    from pg.easybill.all_easybill_x_zoho_x_medisoft
    """)

In [ ]:
duck.sql(
    """
    select 
        count(*) filter(where id_medisoft is not null and id_zoho is not null) as both_matched,
        count(*) filter(where id_medisoft is not null) as medisoft_matched,
        count(*) filter(id_zoho is not null) as zoho_matched,
    from pg.easybill.all_easybill_x_zoho_x_medisoft
    """)

In [ ]:
duck.sql(
    """
    select 
        count(*) filter(where id_medisoft is not null and id_zoho is not null) as both_matched,
        count(*) filter(where id_medisoft is not null) as medisoft_matched,
        count(*) filter(id_zoho is not null) as zoho_matched,
    from pg.easybill.all_easybill_x_zoho_x_medisoft
    """)

# Export results

In [ ]:
duck.sql("from pg.easybill.all_easybill_x_zoho_x_medisoft").to_csv(f"{OUTPUT_DIR / 'all_easybill_x_zoho_x_medisoft_2.csv'}")

# Test on how to split the pfad

In [ ]:
duck.sql(
    """
    select 
        clean_account_name(
            case 
                when array_length(split(pfad, '/')) >= 3 and trim(split(pfad, '/')[2]) = 'Nicht Kunden' 
                    then coalesce(
                        nullif(split(pfad, '/')[3], ''), 
                        nullif(name, ''), 
                        kuerzel
                    )
                else coalesce(
                        nullif(split(pfad, '/')[2], ''), 
                        nullif(split(pfad, '/')[1], ''), 
                        nullif(name, ''), 
                        kuerzel
                    )
            end
        ) as medisoft_clean_name,

        trim(case 
            when array_length(split(pfad, '/')) >= 3 and trim(split(pfad, '/')[2]) = 'Nicht Kunden' 
                then coalesce(
                    nullif(split(pfad, '/')[3], ''), 
                    nullif(name, ''), 
                    kuerzel
                )
            else coalesce(
                    nullif(split(pfad, '/')[2], ''), 
                    nullif(split(pfad, '/')[1], ''), 
                    nullif(name, ''), 
                    kuerzel
                )
        end) as medisoft_name,
        split(pfad, '/') as pfad_split,
    from pg.medisoft.table_firmenstruktur
    where rec_id = '00_8NR00RSGE6'
    """
)

# Match n#2 on medisoft, taking name in priority over the pfad

In [ ]:
duck.sql("""
    with easybill_data as (
        select 
            id_easybill, id_zoho, id_medisoft,
            easybill_name,
            easybill_clean_name
        from pg.easybill.all_easybill_x_zoho_x_medisoft
        where id_medisoft is null
    ), medisoft_data_2 as (
        select
            rec_id as source_id,
            rec_id as id_medisoft,
            clean_account_name(coalesce(name, kuerzel)) as medisoft_clean_name,
            coalesce(name, kuerzel) as medisoft_name,
            pfad
        from pg.medisoft.table_firmenstruktur
        where rec_id not in (
            select 
                distinct id_medisoft 
            from pg.easybill.all_easybill_x_zoho_x_medisoft 
            where id_medisoft is not null 
        )
    ), eb_x_medisoft_matched_firms_2 as (
            select
            *,
            jaro_winkler_similarity(eb.easybill_clean_name, me.medisoft_clean_name) as sim
        from easybill_data eb
        left join medisoft_data_2 me
            on eb.easybill_clean_name = me.medisoft_clean_name
            or jaro_winkler_similarity(eb.easybill_clean_name, me.medisoft_clean_name) > 0.95
        where sim > 0.6
        qualify row_number() over (partition by me.id_medisoft order by sim desc) = 1
        order by eb.id_easybill
    ) select * from eb_x_medisoft_matched_firms_2
    
    """)

In [ ]:
duck.sql(
    """
    select * from pg.medisoft.table_firmenstruktur where rec_id = '00_54Y012ERKE'
    """
)

In [ ]:
duck.sql("select distinct id_medisoft from pg.easybill.all_easybill_x_zoho_x_medisoft")

In [ ]:
duck.sql(
    """
    select 
        --distinct on ("Kontakt: Kundennummer")
        --"Kontakt: Kundennummer",
        "Kontakt: Firma",
        count(*)
    from pg.easybill.contacts
    where 'ejf' in clean_account_name("Kontakt: Firma")
    group by 1
     """)

In [ ]:
duck.sql(
    """
    select count(*) from pg.medisoft.table_firmenstruktur where 'Nicht Kunden' in pfad
    """
)

In [ ]:
duck.sql("select Kundennummer from pg.zoho.Accounts where Kundennummer is not null")